# **CMU 16825 Learning for 3D Vision – Final Project**
## **Uncertainty-Aware Hybrid Rendering with Gaussian Splatting and NeRF for High-Fidelity Synthesis**
### Team 25 Patrick Chen *(Andrew ID: bochunc)*

---

# Abstract

Efficient 3D rendering often faces a trade-off between speed and quality. **3D Gaussian Splatting (GS)** enables efficient rendering but can lose fine details, while **NeRF** offers higher fidelity at significant computational cost. In this project, we propose an **uncertainty-aware hybrid rendering framework** that dynamically combines GS and NeRF predictions. Using a lightweight U-Net model trained on GS features to predict per-pixel uncertainty maps, we selectively re-render the most uncertain regions with NeRF, boosting final image quality without significantly increasing inference time. Experiments show that the hybrid method maintains comparable rendering speed while improving perceptual fidelity (SSIM +0.087, PSNR +1.7dB) compared to GS-only rendering.

---

# Table of Contents
- [Introduction](#introduction)
- [Related Work](#related-work)
- [Method](#method)
- [Neural Uncertainty Feature](#neural-uncertainty-feature)
- [Qualitative Result](#qualitative-result)
- [Quantitative Result](#quantitative-result)
- [References](#references)

---

# Introduction

Rendering 3D scenes typically trades off between efficiency and visual quality. **Gaussian Splatting (GS)** achieves fast rendering speeds but tends to blur fine textures. On the other hand, **Neural Radiance Fields (NeRF)** provide photorealistic quality but are computationally expensive for high-resolution scenes. We aim to develop a **hybrid rendering pipeline** that selectively applies NeRF rendering only to visually uncertain regions, identified through **uncertainty prediction**. This approach combines the advantages of both methods: fast rendering speeds and high-fidelity output, overcoming the limitations faced by either method individually.

---

# Related Work

- **NeRF: Representing Scenes as Neural Radiance Fields for View Synthesis**  
  Introduced novel view synthesis with high-quality results, but suffers from slow inference speed.

- **3D Gaussian Splatting for Real-Time Radiance Field Rendering**  
  Enabled fast 3D scene rendering, but lost high-frequency texture details compared to NeRF.

- **SS Mip-NeRF: Supersampled Mip-NeRF**  
  Enhanced Mip-NeRF’s detail preservation by combining cone-based rendering with supersampling for superior anti-aliasing.

Our project builds upon these works by integrating uncertainty estimation into hybrid rendering for improved trade-off.

---

# Method

<div align="center">

![Uncertainty-Aware Hybrid Rendering Framework](./Figures/Proposed_Uncertainty_Framework.png)
<br>
**Figure 1:** Proposed Uncertainty-Aware Hybrid Rendering Framework.
</div>

We propose an uncertainty-aware hybrid rendering framework that leverages the speed of 3D Gaussian Splatting (GS) and the fidelity of NeRF through uncertainty-guided pixel-wise refinement. As shown in Figure 1, the framework consists of three stages:

**Training Phase:**

- **(1) 3D GS Model Training**:  
  - **Model**: A GS model is trained using multi-view RGB images and their corresponding camera poses.  
  - **Input**: Multi-view RGB images + Camera poses.  
  - **Output**: A trained GS model capable of efficient scene rendering and producing auxiliary features for uncertainty estimation.

- **(2) NeRF Model Training**:  
  - **Model**: A NeRF model is independently trained to predict high-fidelity volume renderings.  
  - **Input**: The same multi-view RGB images + Camera poses.  
  - **Output**: A trained NeRF model capable of high-quality scene synthesis.

- **(3) Neural Uncertainty Model Training**:  
  - **Model**: A lightweight UNet or MLP is trained to predict per-pixel SSIM error based on GS-rendered outputs and GS feature maps.  
  - **Input**: GS features extracted from the GS model (alpha sum, color variance, view direction, and 2D covariance area) + camera poses.  
  - **Output**: An uncertainty prediction model that outputs a per-pixel uncertainty map.

**Inference Phase:**

- **Input during Inference**:  
  - A new camera pose is provided to the pipeline.

- **Steps during Inference**:
  1. **GS Rendering**:  
     - The GS model renders an initial full image and extracts GS feature maps.
  2. **Uncertainty Prediction**:  
     - The uncertainty model takes the GS feature maps and predicts an uncertainty score for each pixel.
  3. **Pixel Selection**:  
     - The top-k% most uncertain pixels are selected for refinement.
  4. **NeRF Refinement**:  
     - The NeRF model re-renders only the selected uncertain pixels based on the input camera pose.
  5. **Merging**:  
     - The final image is composed by combining GS-rendered pixels with NeRF-refined pixels at uncertain regions.

This hybrid approach allows the system to preserve the efficiency of GS while selectively enhancing critical regions using NeRF, achieving a balance between fast rendering and high visual quality.

---

# Neural Uncertainty Feature

<div align="center">

![Feature Extraction for Neural Unceratainty Prediction](./Figures/Feature_Extraction.png)
<br>
**Figure 2:** Feature Extraction for Neural Uncertainty Prediction.
</div>

We extract four types of features from the GS model for uncertainty prediction. The input to the feature extraction consists of the output from the trained 3D Gaussian Splatting (GS) model, which includes the parameters of each 3D splat: 3D mean position, orientation quaternion, opacity (alpha), scales (radii), and RGB color values. These attributes are processed together with the known camera pose to generate per-pixel feature maps over the rendered image plane. To be more specific, a 2D Gaussian is represented by the following expression:

$$
f(x; \mu_i, \Sigma_i) = \frac{1}{2\pi \sqrt{|\Sigma_i|}} \exp\left( -\frac{1}{2} (x-\mu_i)^T \Sigma_i^{-1} (x-\mu_i) \right)
$$

where:
- $x$ is a 2D vector that represents the pixel location
- $\mu_i$ is the 2D vector representing the mean of the $i$-th 2D Gaussian
- $\Sigma_i$ is the covariance of the 2D Gaussian

Given the opacity $o_i$ of a 3D Gaussian, the alpha value at pixel $x$ is computed as:

$$
\alpha_i(x) = o_i \exp\left(P(x,i)\right)
$$

where

$$
P(x,i) = -\frac{1}{2} (x-\mu_i)^T \Sigma_i^{-1} (x-\mu_i)
$$

Thus, $\alpha_i(x)$ represents the contribution of the $i$-th Gaussian to the pixel opacity at location $x$. 

We extract the four feature maps as the following for training our uncertainty prediction model:


- **Alpha sum**: Measures the total accumulated opacity at each pixel.

  $$
  \text{Alpha Sum}(x, y) = \sum_{i=1}^{N} \alpha_i(x, y)
  $$

  The alpha sum indicates the overall visibility along the viewing ray at a given pixel. Pixels with low accumulated opacity (e.g., transparent or sparsely covered regions) often suffer from high rendering uncertainty due to insufficient splat contributions. Conversely, very dense opacity could cause over-blending and loss of sharpness. Thus, alpha sum provides a strong signal for identifying uncertain regions.


- **Color variance**: Reflects the RGB variance of splat colors weighted by transmittance and opacity.

  $$
  \text{Color Variance}(x, y) = \frac{1}{3} \sum_{c \in \{R,G,B\}} \left( \frac{ \sum_{i=1}^{N} w_i(x,y) (c_i(x,y) - \bar{c}(x,y))^2 }{ \sum_{i=1}^{N} w_i(x,y) } \right)
  $$

  where

  $$
  w_i(x, y) = \alpha_i(x, y) \times \text{transmittance}_i(x, y),
  $$

  and

  $$
  \bar{c}(x,y) = \frac{\sum_{i=1}^{N} w_i(x,y) \, c_i(x,y)}{\sum_{i=1}^{N} w_i(x,y)}
  $$

  Here, $\bar{c}(x,y)$ represents the weighted average color at pixel $(x,y)$, computed using the weights from opacity and transmittance. It serves as the local mean color, allowing us to measure how much each splat's color deviates from the expected average appearance at that pixel. A high deviation indicates the presence of multiple surfaces, lighting variations, or texture discontinuities, leading to greater uncertainty. Therefore, regions with large color variance are more likely to suffer from rendering errors and benefit from NeRF-based refinement.
  


- **2D covariance area**: Captures the average projected footprint size (area of splat spread).

  $$
  \text{2D Covariance Area}(x, y) = \frac{ \sum_{i=1}^{N} \alpha_i(x, y) \det(\Sigma_i) }{ \sum_{i=1}^{N} \alpha_i(x, y) }
  $$

  where $\Sigma_i$ is the 2D covariance matrix of the $i$-th splat.

  The 2D covariance area measures how spread out the projected splats are on the image plane. Larger footprint areas imply a higher degree of depth uncertainty, motion blur, or blending between objects. Pixels associated with large projected areas tend to be more uncertain due to ambiguities in fine geometry or appearance, making this feature a reliable predictor of error-prone regions.


- **View direction cosine**: Represents the weighted cosine of the viewing angle.

  $$
  \text{View Direction Cosine}(x, y) = \frac{ \sum_{i=1}^{N} \alpha_i(x, y) \cos(\theta_i) }{ \sum_{i=1}^{N} \alpha_i(x, y) }
  $$

  where $\theta_i$ is the angle between the camera viewing direction and the splat’s surface orientation.

  The view direction cosine quantifies how oblique the viewing angle is relative to the splat’s normal. Pixels viewed at grazing angles often exhibit foreshortening, lower surface visibility, and higher rendering distortion. As such, highly oblique view angles are strongly correlated with greater rendering uncertainty, making this an important geometric feature for uncertainty prediction.


These four feature maps, each of size $[H, W]$, totally $[H, W, 4]$, are fed into a neural network (MLP or UNet) to predict per-pixel SSIM-based uncertainty.

---

### Implemented Model: Dynamic Graph CNN (DGCNN)
I implemented **DGCNN (Dynamic Graph CNN)** to incorporate local geometric features by constructing edge features via k-nearest neighbors (k-NN) in feature space. Unlike vanilla PointNet, DGCNN dynamically builds local graphs to better capture fine-grained geometric structures.

Reference: [DGCNN Paper](https://arxiv.org/pdf/1801.07829.pdf)

---

### Classification Results Comparison

| Model       | Number of Points | Training Epochs | Overall Test Accuracy |
|-------------|------------------|-----------------|------------------------|
| PointNet (Task 1)  | 10000      | 250       | 97.90%                 |
| **DGCNN**       | 10000        | **201**      | **98.95%**             |


> DGCNN classification accuracy outperforms PointNet by **+1.05%** under the same point count but with fewer training epochs.

---

### Segmentation Results Comparison

| Model       | Number of Points | Training Epochs | Overall Test Accuracy |
|-------------|------------------|-----------------|------------------------|
| PointNet (Task 2)  | 10000      | 300       | 90.83%                 |
| **DGCNN**       | 10000        | **251**      | **92.24%**             |

> DGCNN segmentation accuracy outperforms PointNet by **+1.41%** under the same point count but with fewer training epochs.

---

### Classification Visualization (Qualitative Comparison)

Below are classification visualizations on representative samples from the test set:


| Ground Truth Class | PointNet (Task 1) Predicted Class | PointNet (Task 1) Rendered Point Cloud GIF |  DGCNN Predicted Class | DGCNN Rendered Point Cloud GIF |
|--------------------|------------------|----|----|---------------------------------------------|
| Lamp              | Lamp            | ![1](./output/cls_vis_rotate0/correct_0_lamp_idx_806.gif)   | Lamp | ![11](./output_dgcnn/cls_vis/examine_0_lamp_idx_806.gif) |
| Chair              | Chair            | ![2](./output/cls_vis_rotate0/correct_1_chair_idx_333.gif)   | Chair | ![22](./output_dgcnn/cls_vis/examine_1_chair_idx_333.gif) |
| Vase              | Vase            | ![3](./output/cls_vis_rotate0/correct_2_vase_idx_690.gif)   | Vase | ![33](./output_dgcnn/cls_vis/examine_2_vase_idx_690.gif) |
| Chair               | Lamp             | ![6](./output/cls_vis_rotate0/fail_GT_chair_PRED_lamp_idx_406.gif) |  Chair | ![34](./output_dgcnn/cls_vis/examine_3_chair_idx_406.gif)    |
| Lamp              | Vase             | ![7](./output/cls_vis_rotate0/fail_GT_lamp_PRED_vase_idx_750.gif)    |  Vase | ![35](./output_dgcnn/cls_vis/examine_4_vase_idx_750.gif) |
| Vase              | Lamp             | ![8](./output/cls_vis_rotate0/fail_GT_vase_PRED_lamp_idx_650.gif) | Vase  |![36](./output_dgcnn/cls_vis/examine_5_vase_idx_650.gif)|


> DGCNN shows clearer decision boundaries and better geometric awareness.

> Misclassifications by PointNet (e.g., Chair → Lamp, Lamp → Vase) are often corrected by DGCNN, thanks to its edge-based local feature modeling.

> DGCNN better preserves local part boundaries and predicts the correct labels in challenging examples where PointNet misclassifies.

---

### Segmentation Visualization (Qualitative Comparison)

Below are segmentation visualizations on representative samples from the test set:

| Object Case         | PointNet (Task 2) Prediction Accuracy | PointNet (Task 2) Prediction Result |DGCNN Prediction Accuracy| DGCNN Prediction Result| Ground Truth |
|---------------------|---------------------|--------------------|--------------|-----|-----|
| good case 1   | 99.50%               |  ![gc1](./output/seg_vis/correct_616_pred.gif) |99.56%| ![dgcnn1](./output_dgcnn/seg_vis/examine_correct_616_idx_616_pred.gif)            | ![gcg1](./output/seg_vis/correct_616_gt.gif)       |
| good case 2    | 99.56%               | ![gc2](./output/seg_vis/correct_562_pred.gif)   |99.56%|![dgcnn2](./output_dgcnn/seg_vis/examine_correct_562_idx_562_pred.gif)|            ![gcg2](./output/seg_vis/correct_562_gt.gif)         |
| good case 3    | 99.57%               |  ![gc3](./output/seg_vis/correct_397_pred.gif)   |99.39%|![dgcnn3](./output_dgcnn/seg_vis/examine_correct_397_idx_397_pred.gif)|           ![gcg3](./output/seg_vis/correct_397_gt.gif)         |
| bad case 1 | 43.77%        |   ![bc1](./output/seg_vis/fail_235_pred.gif)    |44.89%| ![dgcnn4](./output_dgcnn/seg_vis/examine_fail_235_idx_235_pred.gif)        | ![bcg1](./output/seg_vis/fail_235_gt.gif)        |
| bad case 2  | 48.19%        |  ![bc2](./output/seg_vis/fail_26_pred.gif)       |53.10%|![dgcnn5](./output_dgcnn/seg_vis/examine_26_idx_26_pred.gif)|       ![bcg2](./output/seg_vis/fail_26_gt.gif)         |

> DGCNN consistently predicts finer part-level structures than PointNet, especially in complex object geometries.

> Even in low-performing examples (bad cases), DGCNN tends to preserve more part consistency across symmetric structures (e.g., chair legs).

> DGCNN demonstrates enhanced local structural sensitivity, leading to smoother and more accurate segment boundaries in 3D point clouds compared to PointNet.

---

### Interpretation
DGCNN outperforms PointNet in both classification and segmentation tasks by leveraging dynamic graph-based local features, which better capture fine-grained structural relationships in 3D point clouds. Its advantage is particularly evident in objects with complex geometries or repeated structures, where preserving local continuity is crucial. However, DGCNN’s reliance on nearest-neighbor graphs makes it more sensitive to noise, sparsity, or irregular point distribution, which may introduce instability in graph construction and degrade performance in certain cases. For example, bad cases show that when local neighborhood consistency is broken, like the chair legs in bad case 1 and the chair head in bad case 2, the edge feature extraction may become unreliable, resulting in lower accuracy or incorrect boundaries. This is because when features are aggregated via edge convolutions and global pooling, dense parts like the seat dominate, while sparse parts like legs contribute little. Besides, the misleading local features, like position, color, and shape, of heads and those from nearby regions like the torso are similar, resulting in feature confusion in mulitple layers of KNN graph edge convolution. Overall, DGCNN's locality-aware design enhances geometric understanding but requires careful preprocessing or robustness enhancements to handle degraded inputs effectively.